# 02 — Theory Validation

Notebook 01 measured the null. Notebook 03 measures performance. **This one checks
that the theory the whole project rests on is implemented correctly**, by testing
each result against simulation where the answer is known in closed form.

Five results, in dependency order:

1. **Marchenko–Pastur** — the bulk density and its edges $\lambda_\pm = \sigma^2(1\pm\sqrt q)^2$
2. **The Stieltjes transform** — the empirical resolvent against the analytic MP transform, and what $\eta$ costs
3. **The BBP spike transition** — where an outlier appears, and how much of its eigenvector survives
4. **Tracy–Widom** — the fluctuation scale of the largest bulk eigenvalue, giving a *principled* spike threshold to replace the hard cutoff in notebook 01
5. **Ledoit–Péché** — the cleaning formula verified pointwise against the computable oracle

Then a non-asymptotic cross-check: cross-validated eigenvalue shrinkage, which
never invokes large-$N$ limits and so tests whether the asymptotics are actually
biting at your $N$.

Pure simulation. No data, no portfolios. Every claim here is falsifiable in one cell.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kstest

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.spectral import (spectrum, mp_pdf, mp_cdf, mp_edges, fit_mp_bulk,
                          stieltjes, default_eta)
from src.estimators import build, invert_spike, spike_overlap, _lp_formula
from src.data import simulate_returns, factor_correlation

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (7, 4),
                     "axes.grid": True, "grid.alpha": .25, "font.size": 9})
FIGDIR = ROOT / "results" / "figures"; FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = ROOT / "results" / "tables";  TABDIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(11)

def white(N, T, rng=None):
    '''Standardised white sample: population correlation is exactly the identity.'''
    rng = RNG if rng is None else rng
    X = rng.standard_normal((T, N))
    return (X - X.mean(0)) / X.std(0, ddof=1)

CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append(dict(test=name, result="PASS" if passed else "FAIL", detail=detail))
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}  {detail}")

## 1 — Marchenko–Pastur

$$\rho(\lambda) = \frac{\sqrt{(\lambda_+-\lambda)(\lambda-\lambda_-)}}{2\pi q \sigma^2 \lambda},
\qquad \lambda_\pm = \sigma^2(1\pm\sqrt q)^2$$

For $C = I$ the answer is known exactly, so this is a pure implementation test:
the empirical edges must land on the analytic ones and the bulk CDF must match.
Convergence should be visible as $N$ grows — that is the asymptotic statement
doing its job.

In [ ]:
rows = []
for q in [0.2, 0.5, 0.8]:
    for N in [100, 200, 400, 800]:
        T = int(round(N / q))
        ev, _, _ = spectrum(white(N, T))
        lo, hi = mp_edges(q)
        ks = kstest(ev, lambda x: mp_cdf(x, q)).statistic
        rows.append(dict(q=q, N=N, T=T, lam_max=ev[-1], lam_plus=hi,
                         edge_err=(ev[-1] - hi) / hi,
                         lam_min=ev[0], lam_minus=lo, ks=ks))
mp_tab = pd.DataFrame(rows)
print(mp_tab.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

worst_ks = mp_tab.query("N >= 400")["ks"].max()
edge = mp_tab.query("N >= 400")["edge_err"].abs().max()
print()
check("MP bulk CDF matches (N>=400)", worst_ks < 0.05, f"max KS = {worst_ks:.4f}")
check("upper edge within 3% (N>=400)", edge < 0.03, f"max rel error = {edge:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, q in zip(axes, [0.2, 0.5, 0.8]):
    N = 800; T = int(N / q)
    ev, _, _ = spectrum(white(N, T))
    ax.hist(ev, bins=70, density=True, alpha=.5, color="C0")
    g = np.linspace(*mp_edges(q), 500)
    ax.plot(g, mp_pdf(g, q), "k-", lw=1.8)
    ax.set_title(f"q = {q}"); ax.set_xlabel("$\\lambda$")
axes[0].set_ylabel("density")
fig.suptitle("Marchenko-Pastur, $C = I$, N = 800", y=1.03)
fig.tight_layout(); fig.savefig(FIGDIR / "02_mp_law.png", bbox_inches="tight"); plt.show()

## 2 — The Stieltjes transform and the cost of $\eta$

$$m(z) = \frac1N \operatorname{Tr}(z\mathbb I - E)^{-1} = \frac1N \sum_j \frac{1}{z - \lambda_j}$$

The empirical $m$ is a comb of poles; the limiting one is smooth. For MP the limit
has a closed form,

$$m(z) = \frac{(z+q-1) - \sqrt{(z-q-1)^2 - 4q}}{2qz}$$

so we can measure directly what the regularisation $\eta$ costs. This is the only
real numerical choice in the project, and the $\eta$-sensitivity curve below is why
the RIE uses a *relative* shift $z = \lambda(1-i\eta)$: an absolute shift misbehaves
once $\sigma^2$ drifts away from 1.

In [ ]:
def mp_stieltjes(z, q):
    '''Analytic MP Stieltjes transform.

    m solves q z m^2 - (z + q - 1) m + 1 = 0, which has two roots.  numpy's
    principal square root picks the wrong one over part of the bulk, so select
    by the defining property instead: m(z) = int rho(x)/(z-x) dx implies
    sign(Im m) = -sign(Im z).  Getting this wrong is silent -- the magnitudes
    look plausible and only the sign of the imaginary part is flipped.
    '''
    z = np.atleast_1d(np.asarray(z, complex))
    disc = np.sqrt((z - q - 1) ** 2 - 4 * q + 0j)
    r1 = ((z + q - 1) - disc) / (2 * q * z)
    r2 = ((z + q - 1) + disc) / (2 * q * z)
    good = np.sign(r1.imag) == -np.sign(z.imag)
    return np.where(good, r1, r2)

q, N = 0.5, 1000
ev, _, _ = spectrum(white(N, int(N / q)))
grid = np.linspace(mp_edges(q)[0] + .02, mp_edges(q)[1] - .02, 200)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
errs = []
for eta_rel, c in [(0.01, "C0"), (0.05, "C2"), (0.15, "C3"), (0.40, "C4")]:
    z = grid * (1 - 1j * eta_rel)
    emp, ana = stieltjes(ev, z), mp_stieltjes(z, q)
    axes[0].plot(grid, -emp.imag / np.pi, color=c, lw=1.2, label=f"$\\eta$ = {eta_rel}")
    errs.append(dict(eta=eta_rel, med_rel_err=float(np.median(np.abs(emp - ana) / np.abs(ana)))))
axes[0].plot(grid, mp_pdf(grid, q), "k--", lw=1.8, label="true density")
axes[0].set_xlabel("$\\lambda$"); axes[0].set_ylabel("$-\\mathrm{Im}\\, m/\\pi$")
axes[0].set_title("Recovered density vs regularisation"); axes[0].legend(fontsize=7.5)

sens = pd.DataFrame(errs).set_index("eta")
fine = []
for e_ in np.logspace(-3, -0.3, 25):
    z = grid * (1 - 1j * e_)
    fine.append(np.median(np.abs(stieltjes(ev, z) - mp_stieltjes(z, q)) / np.abs(mp_stieltjes(z, q))))
axes[1].loglog(np.logspace(-3, -0.3, 25), fine, "o-", ms=3, color="C3")
axes[1].axvline(default_eta(N), color="k", ls=":", lw=1.2, label=f"$N^{{-1/2}}$ = {default_eta(N):.3f}")
axes[1].set_xlabel("$\\eta$"); axes[1].set_ylabel("median relative error in $m$")
axes[1].set_title("Bias-variance trade-off in $\\eta$"); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR / "02_stieltjes_eta.png", bbox_inches="tight"); plt.show()

best = np.logspace(-3, -0.3, 25)[int(np.argmin(fine))]
print(sens.to_string(float_format=lambda v: f"{v:9.4f}"))
check("empirical m matches analytic MP at moderate eta",
      min(fine) < 0.05, f"min rel err = {min(fine):.4f} at eta = {best:.3f}")
z0 = grid * (1 - 1j * default_eta(N))
err0 = float(np.median(np.abs(stieltjes(ev, z0) - mp_stieltjes(z0, q)) / np.abs(mp_stieltjes(z0, q))))
check("relative error already small at the N^-1/2 default", err0 < 0.05,
      f"{err0:.4f} at eta = {default_eta(N):.3f}")
print("\nNote the error curve is monotone decreasing in eta. That is expected and"
      "\nnot an argument for large eta: here the truth IS the smooth MP limit, so"
      "\nmore smoothing can only help. On a spectrum with real structure, smoothing"
      "\nerases the features you are trying to estimate. Resolvent accuracy against"
      "\nthe limit is therefore the wrong criterion for choosing eta -- the"
      "\ndownstream oracle-recovery curve in notebook 03 is the right one.")

## 3 — The BBP spike transition

A population spike $\theta$ pushes its sample eigenvalue to

$$\lambda = \theta\left(1 + \frac{q\sigma^2}{\theta-\sigma^2}\right)$$

and the squared overlap between the sample and population eigenvectors is

$$\omega = \frac{1 - q\sigma^4/(\theta-\sigma^2)^2}{1 + q\sigma^2/(\theta-\sigma^2)}$$

Both vanish into the bulk below the detectability threshold $\theta = \sigma^2(1+\sqrt q)$.
This is the single most important limitation of the whole framework: below the
threshold there is *no information left to recover*, by any estimator.

In [ ]:
q, N = 0.5, 600
T = int(N / q)
thetas = np.concatenate([np.linspace(1.05, 1 + np.sqrt(q), 6),
                         np.linspace(1 + np.sqrt(q) + .05, 6.0, 12)])
obs_lam, obs_om, n_rep = [], [], 12

for th in thetas:
    ls, os_ = [], []
    for _ in range(n_rep):
        v = RNG.standard_normal(N); v /= np.linalg.norm(v)
        C = np.eye(N) + (th - 1) * np.outer(v, v)
        d = np.sqrt(np.diag(C)); C = C / np.outer(d, d)
        X = simulate_returns(C, T, rng=RNG); X = (X - X.mean(0)) / X.std(0, ddof=1)
        ev, U, _ = spectrum(X)
        ls.append(ev[-1]); os_.append(float(U[:, -1] @ v) ** 2)
    obs_lam.append(np.mean(ls)); obs_om.append(np.mean(os_))

th_grid = np.linspace(1.001, 6, 400)
pred_lam = np.where(th_grid > 1 + np.sqrt(q),
                    th_grid * (1 + q / (th_grid - 1)), (1 + np.sqrt(q)) ** 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(th_grid, pred_lam, "k-", lw=1.8, label="theory")
axes[0].plot(thetas, obs_lam, "o", ms=5, color="C3", label="simulation")
axes[0].axhline((1 + np.sqrt(q)) ** 2, color="C0", ls="--", lw=1, label="$\\lambda_+$")
axes[0].axvline(1 + np.sqrt(q), color="k", ls=":", lw=1.2)
axes[0].set_xlabel("population spike $\\theta$"); axes[0].set_ylabel("largest sample $\\lambda$")
axes[0].set_title("Outlier location"); axes[0].legend(fontsize=8)

axes[1].plot(th_grid, spike_overlap(th_grid, q), "k-", lw=1.8, label="theory")
axes[1].plot(thetas, obs_om, "o", ms=5, color="C3", label="simulation")
axes[1].axvline(1 + np.sqrt(q), color="k", ls=":", lw=1.2,
                label=f"BBP threshold $1+\\sqrt{{q}}$ = {1+np.sqrt(q):.2f}")
axes[1].set_xlabel("$\\theta$"); axes[1].set_ylabel("squared overlap $\\omega$")
axes[1].set_title("Eigenvector recovery"); axes[1].legend(fontsize=8)
fig.suptitle(f"BBP transition, N = {N}, q = {q}", y=1.03)
fig.tight_layout(); fig.savefig(FIGDIR / "02_bbp_transition.png", bbox_inches="tight"); plt.show()

above = thetas > 1 + np.sqrt(q)
lam_err = np.abs(np.array(obs_lam)[above] -
                 np.array(thetas)[above] * (1 + q / (np.array(thetas)[above] - 1))) / np.array(obs_lam)[above]
om_err = np.abs(np.array(obs_om)[above] - spike_overlap(np.array(thetas)[above], q))
check("outlier location matches BBP", lam_err.max() < 0.05, f"max rel err = {lam_err.max():.4f}")
check("eigenvector overlap matches BBP", om_err.max() < 0.08, f"max abs err = {om_err.max():.4f}")
check("no recovery below threshold", np.mean(np.array(obs_om)[~above]) < 0.25,
      f"mean overlap below = {np.mean(np.array(obs_om)[~above]):.3f}")

In [ ]:
# invert_spike must undo the map it was built from
# Only spikes above the detectability threshold theta = 1 + sqrt(q) are
# invertible: below it the map lambda(theta) is decreasing and the outlier has
# already merged into the bulk, so there is nothing to recover.
print(f"detectability threshold: theta > 1 + sqrt(q) = {1 + np.sqrt(q):.4f}\n")
th_true = np.array([2.0, 2.5, 4.0, 8.0])
lam_sim = th_true * (1 + q / (th_true - 1))
th_hat, ok = invert_spike(lam_sim, q, 1.0)
print(pd.DataFrame({"theta_true": th_true, "lambda": lam_sim,
                    "theta_recovered": th_hat, "detectable": ok})
      .to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
check("invert_spike inverts the BBP map",
      np.max(np.abs(th_hat - th_true) / th_true) < 1e-6,
      f"max rel err = {np.max(np.abs(th_hat - th_true)/th_true):.2e}")

## 4 — Tracy–Widom and a principled spike threshold

Notebook 01 counts spikes by thresholding at $\lambda_+$, which over-counts: the
largest *bulk* eigenvalue fluctuates around $\lambda_+$ on the scale $N^{-2/3}$, so
a crossing or two is expected under the pure null.

Rather than import a Tracy–Widom table, build the null by Monte Carlo — simulate
white Wisharts at the same $(N,T)$ and take the empirical distribution of
$\lambda_{\max}$. Its 99th percentile is a calibrated threshold, and the width
should scale as $N^{-2/3}$, which is itself a testable prediction.

In [ ]:
def tw_null(N, T, n_sim=300, rng=None):
    rng = RNG if rng is None else rng
    return np.array([spectrum(white(N, T, rng))[0][-1] for _ in range(n_sim)])

q = 0.5
scal = []
for N in [100, 200, 400]:
    lm = tw_null(N, int(N / q), n_sim=200)
    lp = mp_edges(q)[1]
    scal.append(dict(N=N, mean=lm.mean(), lam_plus=lp, sd=lm.std(),
                     sd_times_N23=lm.std() * N ** (2 / 3),
                     p99=np.quantile(lm, .99),
                     excess_p99=(np.quantile(lm, .99) - lp) / lp))
tw = pd.DataFrame(scal)
print(tw.to_string(index=False, float_format=lambda v: f"{v:10.4f}"))

ratio = tw["sd_times_N23"].max() / tw["sd_times_N23"].min()
check("lambda_max fluctuation scales as N^(-2/3)", ratio < 1.6,
      f"sd * N^(2/3) varies by {ratio:.2f}x over N = 100..400")

N = 400; T = int(N / q)
lm = tw_null(N, T, n_sim=300)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(lm, bins=40, density=True, alpha=.55, color="C0", label="$\\lambda_{max}$, white null")
ax.axvline(mp_edges(q)[1], color="k", ls="--", lw=1.5, label="$\\lambda_+$ (MP edge)")
ax.axvline(np.quantile(lm, .99), color="C3", lw=1.8, label="99th pct (calibrated threshold)")
ax.set_xlabel("$\\lambda_{max}$"); ax.set_title(f"Tracy-Widom null, N = {N}, q = {q}")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "02_tracy_widom.png", bbox_inches="tight"); plt.show()

frac_above = (lm > mp_edges(q)[1]).mean()
print(f"\nUnder the pure null, {100*frac_above:.0f}% of samples put lambda_max above "
      f"lambda_+.\nThresholding at the MP edge therefore over-counts spikes by "
      f"construction; the 99th percentile ({np.quantile(lm,.99):.4f}) is the honest cutoff.")

## 5 — Ledoit–Péché, verified pointwise

The theorem that the whole project rests on: the oracle overlap
$\xi_i = u_i^\top C u_i$, which contains the unknown $C$, converges to a function of
the sample spectrum alone,

$$\xi_i \to \frac{\lambda_i}{\left|1 - q + q\lambda_i\, m(\lambda_i - i0^+)\right|^2}$$

On synthetic data both sides are computable, so this is a direct test of the
theorem *and* of the implementation.

In [ ]:
rows = []
for (N, T) in [(200, 1000), (300, 600), (400, 500)]:
    q_ = N / T
    C = factor_correlation(N, k=3, loading_sd=.5, rng=RNG)
    X = simulate_returns(C, T, rng=RNG); X = (X - X.mean(0)) / X.std(0, ddof=1)
    sp = spectrum(X); lam = sp[0]
    f = fit_mp_bulk(lam, q0=q_)
    orc = build("oracle", C_true=C).fit(X, spectrum_=sp).eigenvalues_
    eta = default_eta(N)
    lp = _lp_formula(lam, q_, stieltjes(lam, lam * (1 - 1j * eta)))
    bulk = lam <= f["lambda_plus"]
    rows.append(dict(N=N, T=T, q=q_,
                     med_rel_err=float(np.median(np.abs(lp[bulk] - orc[bulk]) / orc[bulk])),
                     corr=float(np.corrcoef(lp[bulk], orc[bulk])[0, 1]),
                     gap=float(np.linalg.norm(lp[bulk] - orc[bulk]) /
                               np.linalg.norm(lam[bulk] - orc[bulk]))))
    if N == 300:
        keep = (lam, lp, orc, bulk, f)

lp_tab = pd.DataFrame(rows)
print(lp_tab.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
check("Ledoit-Peche formula tracks the oracle in the bulk",
      lp_tab["gap"].max() < 0.35, f"max residual gap = {lp_tab['gap'].max():.4f}")

lam, lp, orc, bulk, f = keep
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(lam[bulk], lam[bulk], color="C7", ls="--", lw=1, label="sample (identity)")
ax.plot(lam[bulk], orc[bulk], "k+", ms=6, label="oracle $u_i' C u_i$")
ax.plot(lam[bulk], lp[bulk], ".", ms=5, color="C3", label="Ledoit-Peche formula")
ax.set_xlabel("$\\lambda$"); ax.set_ylabel("$\\xi$")
ax.set_title("The theorem, verified: the formula reproduces the oracle without knowing $C$")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "02_ledoit_peche.png", bbox_inches="tight"); plt.show()

## 6 — A non-asymptotic cross-check

Cross-validated eigenvalue shrinkage estimates the same oracle overlap without
ever invoking a large-$N$ limit: split the sample, take eigenvectors from one
half, evaluate them against the covariance of the other. If it agrees with the
RIE, the asymptotics are biting at your $N$. If it beats the RIE at small $N$,
they are not — and that is a finding, not a failure.

It is also the only route to an oracle proxy on *real* data, where $C$ is unknown.

In [ ]:
rows = []
for (N, T) in [(100, 300), (200, 600), (300, 600), (400, 500)]:
    C = factor_correlation(N, k=3, loading_sd=.5, rng=RNG)
    X = simulate_returns(C, T, rng=RNG); X = (X - X.mean(0)) / X.std(0, ddof=1)
    sp = spectrum(X)
    r = {}
    for m, kw in [("sample", {}), ("linear", {}), ("rie", {}), ("nonlinear", {}),
                  ("cv", {}), ("oracle", {"C_true": C})]:
        r[m] = np.linalg.norm(build(m, **kw).fit(X, spectrum_=sp).sigma_ - C, "fro")
    rows.append(dict(N=N, T=T, q=N / T, **r))

cv_tab = pd.DataFrame(rows)
print(cv_tab.to_string(index=False, float_format=lambda v: f"{v:9.3f}"))

eff = lambda m: ((cv_tab["sample"] - cv_tab[m]) / (cv_tab["sample"] - cv_tab["oracle"]))
fig, ax = plt.subplots(figsize=(7, 3.8))
for m, c in [("linear", "C0"), ("rie", "C3"), ("nonlinear", "C1"), ("cv", "C2")]:
    ax.plot(cv_tab["q"], eff(m), "o-", ms=5, color=c, label=m)
ax.axhline(1.0, color="k", ls="--", lw=1, label="oracle")
ax.set_xlabel("q = N/T"); ax.set_ylabel("fraction of oracle gain captured")
ax.set_title("Asymptotic (RIE) vs non-asymptotic (CV) estimation")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "02_cv_vs_rie.png", bbox_inches="tight"); plt.show()

check("CV shrinkage beats the sample covariance everywhere",
      (cv_tab["cv"] < cv_tab["sample"]).all(), "")
check("RIE and CV agree to within 15% in loss",
      (np.abs(cv_tab["rie"] - cv_tab["cv"]) / cv_tab["cv"]).max() < 0.15,
      f"max = {(np.abs(cv_tab['rie']-cv_tab['cv'])/cv_tab['cv']).max():.4f}")

## Validation summary

In [ ]:
summary = pd.DataFrame(CHECKS)
summary.to_csv(TABDIR / "theory_validation.csv", index=False)
mp_tab.to_csv(TABDIR / "mp_convergence.csv", index=False)
tw.to_csv(TABDIR / "tracy_widom_scaling.csv", index=False)
print(summary.to_string(index=False))

n_fail = (summary["result"] == "FAIL").sum()
lines = [
    "",
    f"{len(summary) - n_fail}/{len(summary)} checks passed.",
    "",
    "What this licenses:",
    " - the MP implementation, the resolvent, the spike inversion and the",
    "   Ledoit-Peche formula are all doing what the theory says",
    " - at the N^-1/2 default the empirical resolvent already matches the",
    "   analytic MP transform to within a few percent",
    " - spike counting at the MP edge over-counts by construction; the Monte",
    "   Carlo Tracy-Widom threshold in section 4 is the calibrated replacement",
    "   and should supersede the hard cutoff used in notebook 01",
    "",
    "What it does not license: nothing here says the model fits real returns.",
    "That is notebook 01 (does the null hold?) and notebook 03 (does cleaning pay?).",
]
print(chr(10).join(lines))